# Part 4 — From predictions to “what should we do?”

Part 3 gave metrics, plots, and a champion model. This notebook starts turning what we have into what we can do with it.


### Imports


In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
print("Libraries loaded")


Libraries loaded


### Load Part 3 outputs

Need `datasets/part3_predictions.csv` and `datasets/part3_model_metrics.csv`. Rerun Part 3 if either is missing.


In [3]:
ROOT = Path(".")
PRED_IN = ROOT / "datasets" / "part3_predictions.csv"
METRIC_IN = ROOT / "datasets" / "part3_model_metrics.csv"
ACTION_OUT = ROOT / "datasets" / "part4_policy_actions.csv"

pred = pd.read_csv(PRED_IN, parse_dates=["date"])
metrics = pd.read_csv(METRIC_IN)

print("Pred shape:", pred.shape)
print("Metrics shape:", metrics.shape)
display(metrics)
display(pred.head())


Pred shape: (669, 5)
Metrics shape: (4, 5)


,split,model,MAE,RMSE,R2
0,time,Ridge,4.805272,10.496526,0.285126
1,time,RandomForest,4.450732,10.760362,0.248737
2,spatial,Ridge,4.283879,6.125955,0.452406
3,spatial,RandomForest,2.879850,4.696502,0.678145


,date,PM2_5,openaq_id,pred_ridge,pred_rf
0,2025-05-01,18.60,IT0459A,13.758710,15.178781
1,2025-06-01,12.20,IT0459A,16.295384,18.280877
2,2025-07-01,15.10,IT0459A,15.843904,14.177622
3,2025-08-01,9.96,IT0459A,17.110826,12.724715
4,2025-09-01,9.95,IT0459A,19.722782,14.442277


### Pick champion for downstream use


In [5]:
champion_row = pd.Series({"model": "RandomForest", "split": "time"})
champion = "pred_rf"

print("Champion model (fixed):", champion_row["model"])
print("Prediction column:", champion)

Champion model (fixed): RandomForest
Prediction column: pred_rf


### High-risk threshold (WHO-style)

We flag months where PM2.5 is **at or above a fixed health guideline level**.

We use **WHO (2021) 24-hour AQG for PM2.5: 15 µg/m³** as `risk_threshold`.

**Important caveat:** WHO defines this for **daily** exposure; our data are **monthly means** (µg/m³). Using 15 here is a **simple, explainable screen**, not the official legal exceedance calculation.

If you want the stricter **WHO annual AQG (5 µg/m³)**, change `risk_threshold` to `5.0` (it will flag many more months—say that explicitly in the report).


In [13]:
# WHO (2021) PM2.5 24-hour AQG (µg/m³). Monthly use is illustrative (see markdown above).
risk_threshold = 15.0

pred["high_risk_pred"] = pred[champion] >= risk_threshold
pred["high_risk_actual"] = pred["PM2_5"] >= risk_threshold

print("WHO-style threshold used (µg/m³):", risk_threshold)
print("Predicted high-risk rows:", int(pred["high_risk_pred"].sum()))
print("Actual high-risk rows:", int(pred["high_risk_actual"].sum()))


WHO-style threshold used (µg/m³): 15.0
Predicted high-risk rows: 163
Actual high-risk rows: 195


### High-risk precision / recall (rough)

Sanity check for slides: when we flag high risk, how often is reality high too? Not the main Part 3 metric, but helps the story.


In [15]:
tp = int(((pred["high_risk_pred"] == 1) & (pred["high_risk_actual"] == 1)).sum())
fp = int(((pred["high_risk_pred"] == 1) & (pred["high_risk_actual"] == 0)).sum())
fn = int(((pred["high_risk_pred"] == 0) & (pred["high_risk_actual"] == 1)).sum())

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0

print("TP:", tp, "FP:", fp, "FN:", fn)
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))


TP: 111 FP: 52 FN: 84
Precision: 0.681
Recall: 0.569


Under the fixed threshold, the model is moderately precise (~68%) and catches ~57% of exceedance months, so it’s useful as a screening / prioritization tool but not as a standalone enforcement trigger without human + contextual checks.